# Week 3 — Adding crystal structure with a pretrained graph neural network (CHGNet)
**Why:** the remaining large errors (e.g. graphite, predicted ~2.8 V instead of ~0.1 V) are *structural*: formulas cannot tell a layered intercalation host from any other carbon or oxide.

**Approach — transfer learning:** instead of training a graph neural network from scratch on only ~2,600 materials, we use **CHGNet**, a graph neural network already trained on 1.5 million DFT calculations from the Materials Project. For every host crystal it produces a 64-number **structure fingerprint** (its learned "crystal feature" vector). We add these fingerprints to the Week 2b features and retrain XGBoost on **the same split**.

**This notebook:**
1. Re-downloads the electrode data with your API key (to get clean crystal structures)
2. Computes CHGNet fingerprints for every host structure (use the **GPU** runtime)
3. Compares three models on the same split: Week 2b features · structure fingerprints only · both combined
4. Checks the structural outliers (graphite …)
5. Saves everything to Drive

**Before running:** Runtime → Change runtime type → **T4 GPU**.

In [ ]:
!pip -q install chgnet mp-api pymatgen xgboost
from google.colab import drive
drive.mount("/content/drive")
DATA_DIR = "/content/drive/MyDrive/electrode_voltage_ml"

In [ ]:
from getpass import getpass
API_KEY = getpass("Materials Project API key: ")

## 1. Get clean crystal structures
We download the electrode collection again (a few seconds) and read it as a plain table so that every host structure can be rebuilt exactly. Then we match each structure to the electrodes used in Week 2b.

In [ ]:
import glob, json, warnings
import numpy as np, pandas as pd
import pyarrow.dataset as ds
from mp_api.client import MPRester
warnings.filterwarnings("ignore")

with MPRester(API_KEY) as mpr:
    rester = getattr(getattr(mpr, "materials", None), "insertion_electrodes", None) or mpr.insertion_electrodes
    _ = rester.search(fields=["id_charge", "id_discharge", "working_ion", "host_structure"])

paths = glob.glob("/root/mp_datasets/**/insertion-electrodes", recursive=True)
print("Local copy:", paths)
dset = ds.dataset(paths[0], format="parquet")
recs = dset.to_table(columns=["id_charge", "id_discharge", "working_ion", "host_structure"]).to_pylist()
struct_by_key = {(str(r["id_charge"]), str(r["id_discharge"]), str(r["working_ion"])): r["host_structure"] for r in recs}
print(len(struct_by_key), "structures available")

In [ ]:
from pymatgen.core import Structure

ION_ORDER = ["Li", "Na", "K", "Mg", "Ca", "Zn"]
feat = pd.read_csv(f"{DATA_DIR}/features_week2b.csv")            # Week 2b features (same row order as Week 2)
clean = pd.read_csv(f"{DATA_DIR}/electrodes_clean.csv")
clean = clean[clean["working_ion"].isin(ION_ORDER)].reset_index(drop=True)
assert (clean["framework_formula"].values == feat["framework"].values).all()

def to_structure(d):
    if d is None:
        return None
    try:
        return Structure.from_dict(d)
    except Exception:
        return None

structures = []
for ic, idd, ion in zip(clean["id_charge"].astype(str), clean["id_discharge"].astype(str), clean["working_ion"]):
    structures.append(to_structure(struct_by_key.get((ic, idd, ion))))
ok = np.array([s is not None for s in structures])
print(f"Matched structures for {ok.sum()} of {len(ok)} electrodes")
sizes = [len(s) for s in structures if s is not None]
print(f"Atoms per host cell: median {int(np.median(sizes))}, max {max(sizes)}")

## 2. CHGNet structure fingerprints
Each host crystal is passed through the pretrained CHGNet graph neural network. We keep its internal 64-number crystal feature vector — a learned summary of the atoms, bonds and their 3D arrangement. Identical hosts are computed only once. On a T4 GPU this takes a few minutes.

In [ ]:
import torch
from chgnet.model import CHGNet
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", device)
chgnet = CHGNet.load()
chgnet = chgnet.to(device)

# unique host structures (many electrodes share the same host)
uniq, index = {}, []
for s in structures:
    if s is None:
        index.append(-1); continue
    key = (s.composition.reduced_formula, len(s), round(s.volume, 3))
    if key not in uniq:
        uniq[key] = (len(uniq), s)
    index.append(uniq[key][0])
uniq_structs = [v[1] for v in sorted(uniq.values(), key=lambda t: t[0])]
print(len(uniq_structs), "unique host structures")

In [ ]:
from tqdm.auto import tqdm
fps = np.full((len(uniq_structs), 64), np.nan)
batch = 32
for start in tqdm(range(0, len(uniq_structs), batch)):
    chunk = uniq_structs[start:start + batch]
    try:
        preds = chgnet.predict_structure(chunk, task="e", return_crystal_feas=True, batch_size=len(chunk))
        if isinstance(preds, dict): preds = [preds]
        for j, p in enumerate(preds):
            fps[start + j] = np.asarray(p["crystal_fea"]).ravel()[:64]
    except Exception:
        for j, s in enumerate(chunk):          # fall back to one at a time if a batch fails
            try:
                p = chgnet.predict_structure(s, task="e", return_crystal_feas=True)
                fps[start + j] = np.asarray(p["crystal_fea"]).ravel()[:64]
            except Exception:
                pass
print("Fingerprints computed for", int((~np.isnan(fps).any(axis=1)).sum()), "of", len(uniq_structs), "hosts")

FP = pd.DataFrame([fps[i] if i >= 0 else np.full(64, np.nan) for i in index],
                  columns=[f"chg_{k}" for k in range(64)])
FP.to_csv("chgnet_fingerprints.csv", index=False)

## 3. Same split, three models

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

week2 = json.load(open(f"{DATA_DIR}/xgb_params.json"))
y = feat["average_voltage"].values
ions = feat["working_ion"].values
groups = feat["framework"].values
X_2b = feat.drop(columns=["average_voltage", "working_ion", "framework"])
ion_cols = [c for c in X_2b.columns if c.startswith("ion_")] + ["frac_ion_discharge"]
X_struct = pd.concat([FP, X_2b[ion_cols]], axis=1)          # structure fingerprint + ion info only
X_both = pd.concat([X_2b, FP], axis=1)                       # everything

tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42).split(X_2b, y, groups))
def scores(t, p):
    return {"MAE (V)": mean_absolute_error(t, p), "RMSE (V)": np.sqrt(mean_squared_error(t, p)), "R2": r2_score(t, p)}
def fit_predict(X):
    m = XGBRegressor(**week2["best_params"], min_child_weight=2, reg_lambda=1.0, random_state=0, n_jobs=-1)
    m.fit(X.iloc[tr], y[tr]); return m, m.predict(X.iloc[te])

models, preds = {}, {}
for name, X in [("Week 2b (composition + oxidation)", X_2b),
                ("CHGNet structure + ion", X_struct),
                ("Combined (2b + CHGNet)", X_both)]:
    models[name], preds[name] = fit_predict(X)
comp = pd.DataFrame({k: scores(y[te], p) for k, p in preds.items()}).T.round(3)
comp

In [ ]:
rows = []
for i in ION_ORDER:
    m = ions[te] == i
    if m.sum() > 5:
        r = {"ion": i, "n_test": int(m.sum())}
        for k, p in preds.items():
            r[k] = mean_absolute_error(y[te][m], p[m])
        rows.append(r)
per_ion = pd.DataFrame(rows).set_index("ion").round(3)
print("Test MAE (V) per ion:"); per_ion

## 4. Did structure fix the structural outliers?

In [ ]:
err = pd.DataFrame({"framework": groups[te], "ion": ions[te], "DFT_V": y[te]})
for k, p in preds.items():
    err[k] = p
base = "Week 2b (composition + oxidation)"
err["err_2b"] = (err["DFT_V"] - err[base]).abs()
err["err_combined"] = (err["DFT_V"] - err["Combined (2b + CHGNet)"]).abs()
print("Graphite and other carbon hosts:")
display(err[err["framework"] == "C"].round(2))
print("\nThe 10 worst cases of Week 2b, before and after adding structure:")
err.sort_values("err_2b", ascending=False).head(10).round(2)

In [ ]:
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid")
PALETTE = dict(zip(ION_ORDER, sns.color_palette("tab10", len(ION_ORDER))))
fig, ax = plt.subplots(1, 3, figsize=(16, 5.2), sharex=True, sharey=True)
for a, (k, p) in zip(ax, preds.items()):
    for i in ION_ORDER:
        m = ions[te] == i
        a.scatter(y[te][m], p[m], s=10, alpha=0.7, color=PALETTE[i], label=i)
    a.plot([-0.2, 6.2], [-0.2, 6.2], "k--", lw=1)
    a.set(xlim=(-0.2, 6.2), ylim=(-0.2, 6.2), xlabel="DFT average voltage (V)",
          title=f"{k}\nMAE = {mean_absolute_error(y[te], p):.2f} V")
ax[0].set_ylabel("Predicted voltage (V)"); ax[2].legend(title="Ion")
plt.tight_layout(); plt.savefig("fig_parity_week3.png", dpi=200); plt.show()

## 5. Save

In [ ]:
import shutil
models["Combined (2b + CHGNet)"].save_model("xgb_voltage_week3_combined.json")
err.to_csv("test_predictions_week3.csv", index=False)
comp.to_csv("results_week3.csv"); per_ion.to_csv("results_per_ion_week3.csv")
for f in ["chgnet_fingerprints.csv", "xgb_voltage_week3_combined.json", "test_predictions_week3.csv",
          "results_week3.csv", "results_per_ion_week3.csv", "fig_parity_week3.png"]:
    shutil.copy(f, DATA_DIR)
print("Saved to", DATA_DIR)

## What to send me
1. The **matched structures** line and the **fingerprints computed** line
2. The **three-model comparison table**
3. The **per-ion table** and the **graphite / worst-cases** tables
4. The parity plot

**Honest expectations:** with ~2,600 materials, structure fingerprints alone may be *worse* than the Week 2b chemistry features (CHGNet was trained to predict energies, not voltages). The interesting question is whether **combining** them lowers the error further and fixes structural cases such as graphite.